# Intelli-Speech — TRAIN_V1 (ZF-Net + GMM, version 1)

Runs the v1 experiments on the Colab runtime: baselines → GMM noise classifier → ZF-Net mask
estimator → test-split evaluation → results table → listening check.

**Run order:** cells 1 → 12 from top to bottom. If cell 3 asks for a restart, restart the
kernel and continue from cell 4. Every cell skips work whose outputs already exist, so re-running
is safe.

**Writes only** under `Intelli-Speech-data/experiments/`. The dataset `paired_dataset_v3` is
read-only; the scripts refuse to write inside it. All research logic lives in `intelli_speech/`
and `scripts/`; this notebook only orchestrates them.

In [ ]:
# ============================================================
# CELL 1 | RUNTIME
# ============================================================
import platform, sys
import numpy as np
print("Python:", platform.python_version())
print("NumPy :", np.__version__)
try:
    import torch
    GPU = torch.cuda.is_available()
    print("torch :", torch.__version__, "| CUDA:", GPU, "|", torch.cuda.get_device_name(0) if GPU else "CPU only")
except Exception as exc:
    GPU = False
    print("torch unavailable:", exc)

EPOCHS = 20 if GPU else 8
CROP_SEC = 2.0 if GPU else 1.5
BATCH = 16 if GPU else 8
if not GPU:
    print("WARNING: no GPU. ZF-Net training will be slow (several minutes per epoch). "
          "Runtime > Change runtime type > T4 GPU is recommended.")
print(f"Training plan: epochs={EPOCHS} crop={CROP_SEC}s batch={BATCH}")

In [ ]:
# ============================================================
# CELL 2 | DRIVE MOUNT + REPOSITORY SYNC (explicit pull)
# ============================================================
import os, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
REPO_DIR = Path("/content/Intelli-Speech")
REPO_URL = "https://github.com/niharika9nx/Intelli-Speech.git"

if not (DRIVE_ROOT / "MyDrive").is_dir():
    from google.colab import drive
    drive.mount(str(DRIVE_ROOT))

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"],
                           capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes in the Colab checkout (not pulling over them):\n" + dirty)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())
assert (REPO_DIR / "scripts" / "train_zfnet.py").is_file(), "Checkout is missing the v1 code; pull failed?"
print("v1 code present.")

In [ ]:
# ============================================================
# CELL 3 | INSTALL BASELINE + MODEL ENVIRONMENT
# ============================================================
import importlib.metadata as md
numpy_before = md.version("numpy")
!pip install -q -r requirements-colab.txt
numpy_after = md.version("numpy")
!python scripts/setup_colab_runtime.py
print(f"NumPy before={numpy_before} after={numpy_after}")
if numpy_before != numpy_after:
    print("\n>>> NumPy changed. Restart the kernel now, then continue from CELL 4. <<<")

In [ ]:
# ============================================================
# CELL 4 | CONFIG + DATASET CHECK (read-only)
# ============================================================
import os, sys, json
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech"); os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
try:
    GPU, EPOCHS, CROP_SEC, BATCH
except NameError:  # kernel was restarted after cell 3
    import torch
    GPU = torch.cuda.is_available()
    EPOCHS, CROP_SEC, BATCH = (20, 2.0, 16) if GPU else (8, 1.5, 8)

import librosa, numpy as np
from intelli_speech.dataset import load_metadata, split_counts

D = Path("/content/drive/MyDrive/Intelli-Speech-data/paired_dataset_v3")
EXP = D.parent / "experiments"
R = EXP / "zfnet_gmm_v1"
print("NumPy", np.__version__, "| librosa", librosa.__version__, "| GPU", GPU)

meta = load_metadata(D)
print("rows:", len(meta), "| splits:", split_counts(meta))
assert "noise_environment" in meta.columns, "metadata.csv has no noise_environment column"
print("noise classes:", meta["noise_environment"].value_counts().to_dict())

In [ ]:
# ============================================================
# CELL 5 | BASELINES ON THE TEST SPLIT (Wiener + oracle IRM)
# ============================================================
for method in ["wiener", "oracle-irm"]:
    out = EXP / f"{method}_test"
    if (out / "summary.json").is_file():
        print(f"[skip] {method}: {out}/summary.json exists")
        continue
    !python scripts/evaluate_baselines.py --dataset-dir "{D}" --split test --method {method} --include-pesq --include-stoi

In [ ]:
# ============================================================
# CELL 6 | TRAIN THE GMM NOISE CLASSIFIER
# ============================================================
if (R / "gmm.joblib").is_file():
    print("[skip] gmm.joblib exists")
else:
    !python scripts/train_gmm.py --dataset-dir "{D}" --output-dir "{R}"
report = json.loads((R / "gmm_report.json").read_text())
print("validation accuracy:", round(report["validation_accuracy"], 4), "| by SNR:", report["validation_accuracy_by_snr"])
print("temperature:", report["temperature"])
import pandas as pd
print(pd.DataFrame(report["validation_confusion"]).T.to_string())

In [ ]:
# ============================================================
# CELL 7 | ZF-NET SMOKE RUN (1 epoch, 5 steps, separate directory)
# ============================================================
SMOKE = EXP / "zfnet_gmm_v1_smoke"
!python scripts/train_zfnet.py --dataset-dir "{D}" --output-dir "{SMOKE}" --gmm "{R}/gmm.joblib" --epochs 1 --max-steps-per-epoch 5 --val-limit 10 --batch-size {BATCH} --overwrite

In [ ]:
# ============================================================
# CELL 8 | ZF-NET FULL TRAINING
# ============================================================
if (R / "train_summary.json").is_file():
    print("[skip] training already finished:", (R / "train_summary.json").read_text())
else:
    !python scripts/train_zfnet.py --dataset-dir "{D}" --output-dir "{R}" --epochs {EPOCHS} --crop-sec {CROP_SEC} --batch-size {BATCH} --overwrite
import pandas as pd
print(pd.read_csv(R / "history.csv").to_string(index=False))

In [ ]:
# ============================================================
# CELL 9 | EVALUATE ZF-NET + GMM ON THE TEST SPLIT
# ============================================================
out = EXP / "zfnet-gmm_test"
if (out / "summary.json").is_file():
    print("[skip] zfnet-gmm_test/summary.json exists")
else:
    !python scripts/evaluate_baselines.py --dataset-dir "{D}" --split test --method zfnet-gmm --gmm-checkpoint "{R}/gmm.joblib" --zfnet-checkpoint "{R}/zfnet.pt" --include-pesq --include-stoi

In [ ]:
# ============================================================
# CELL 10 | RESULTS TABLE (test split)
# ============================================================
import pandas as pd
labels = {"wiener": "Wiener (non-oracle)", "zfnet-gmm": "ZF-Net + GMM (v1)",
          "oracle-irm": "Oracle IRM (upper bound, not deployable)"}
rows = []
for method, label in labels.items():
    path = EXP / f"{method}_test" / "metrics.csv"
    if not path.is_file():
        print("missing:", path); continue
    df = pd.read_csv(path)
    for snr, group in [("all", df)] + [(f"{s:g} dB", g) for s, g in df.groupby("target_snr_db")]:
        rows.append({"method": label, "input SNR": snr, "n": len(group),
                     "ΔSNR dB": group["snr_improvement_db"].mean(),
                     "PESQ": group["pesq"].mean() if group["pesq"].notna().any() else float("nan"),
                     "STOI": group["stoi"].mean() if group["stoi"].notna().any() else float("nan")})
table = pd.DataFrame(rows)
print(table.to_markdown(index=False, floatfmt=".3f"))
zsum = EXP / "zfnet-gmm_test" / "summary.json"
if zsum.is_file():
    acc = json.loads(zsum.read_text())["summary"].get("noise_classification_accuracy")
    print(f"\nGMM noise-type accuracy on test: {acc:.3f}")

In [ ]:
# ============================================================
# CELL 11 | LISTENING CHECK (3 test samples)
# ============================================================
from IPython.display import Audio, Markdown, display
from intelli_speech.audio_io import load_audio
from intelli_speech.baselines import wiener_enhance
from intelli_speech.dataset import load_pairs
from intelli_speech.enhancer import Enhancer
from intelli_speech.stft import STFTConfig

enhancer = Enhancer.from_checkpoints(R / "gmm.joblib", R / "zfnet.pt")
for pair in load_pairs(D, split="test")[:6:2]:
    clean, _ = load_audio(pair.clean_path)
    noisy, _ = load_audio(pair.noisy_path)
    out = enhancer.enhance(noisy)
    display(Markdown(f"**{pair.source_id}** | {pair.noise_environment} @ {pair.target_snr_db:g} dB "
                     f"| GMM says **{out.noise_type}** ({out.noise_probs[out.noise_type]:.0%})"))
    for name, audio in [("noisy", noisy), ("Wiener", wiener_enhance(noisy, STFTConfig()).enhanced),
                        ("ZF-Net + GMM", out.waveform), ("clean", clean)]:
        display(Markdown(name)); display(Audio(audio, rate=16000))

## CELL 12 | Use the trained model in the app (local machine)

1. Copy `Intelli-Speech-data/experiments/zfnet_gmm_v1/gmm.joblib` and `zfnet.pt` from Google Drive
   into the repository's git-ignored `checkpoints/` folder on the local machine.
2. Start the inference server from the repository root:
   `INTELLI_SPEECH_GMM_PATH=checkpoints/gmm.joblib INTELLI_SPEECH_ZFNET_PATH=checkpoints/zfnet.pt python -m uvicorn server.app:app --port 8000`
3. Start the app: `cd app && VITE_ENHANCE_API_URL=http://127.0.0.1:8000 npm run dev`

Record the results table from CELL 10 in `PROJECT_NOTES.md`; do not create new Markdown files.